# Incialització

Aquest notebook és un boilerplate per a tots els notebooks posteriors. Aquí es carreguen totes les llibreries, dades i funcions que es necessiten per a les anàlisis i classificacions. D'aquesta manera, si cal canviar alguna cosa en la càrrega de dades o en les funcions, només cal fer-ho aquí i es reflectirà en tots els notebooks que importin aquest. També serveix com a punt de partida per a qualsevol nou notebook que es vulgui crear, ja que ja tindrà tot el necessari carregat i preparat.

In [ ]:
import gc
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from matplotlib.patches import Circle, Ellipse, Rectangle
from scipy.signal import stft
from scipy.stats import ConstantInputWarning, binomtest, chi2, spearmanr
from sklearn.decomposition import FastICA
from sklearn.preprocessing import StandardScaler

# Avisos coneguts de l'entorn EEG.
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module="mne",
    message=".*does not conform to MNE naming conventions.*",
)
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module="mne",
    message=".*events passed to the Epochs constructor are not chronologically ordered.*",
)
mne.set_log_level("WARNING")

print("Inicialització compartida: imports i configuració base carregats.")

In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    candidate_paths = [
        Path("/content/drive/Shareddrives/TFG_Nils_2026"),
        Path("/content/drive/MyDrive/TFG_Nils_2026"),
    ]
    BASE_DIR = next((p for p in candidate_paths if p.exists()), candidate_paths[0])
    DATA_DIR = BASE_DIR / "02_Data" / "01_Preprocessed_epochs"
    META_DIR = BASE_DIR / "02_Data" / "02_meta"
else:
    BASE_DIR = Path(".").resolve()
    DATA_DIR = BASE_DIR / "data"
    META_DIR = BASE_DIR / "docs"

DATA_FILES = sorted(DATA_DIR.glob("*.fif"))

print(f"Entorn: {'Colab' if IN_COLAB else 'Local'}")
print(f"Directori de dades: {DATA_DIR}")
print(f"Fitxers .fif trobats: {len(DATA_FILES)}")

In [ ]:
# Condicions i etiquetes compartides.
COND_EVENTS = {
    "Congruent": ["event_1", "event_2"],
    "Incongruent_Pur": ["event_3", "event_4", "event_5", "event_6"],
    "Incongruent_Mixt": ["event_7", "event_8"],
    "Incongruent": ["event_3", "event_4", "event_5", "event_6", "event_7", "event_8"],
    "Errors": ["event_9", "event_10", "event_11", "event_12"],
}

EVENT_TO_LABEL = {
    1: "CRP",
    2: "CGP",
    3: "IPR",
    4: "IPR",
    5: "IPG",
    6: "IPG",
    7: "IM",
    8: "IM",
    9: "CRG",
    10: "CGG",
    11: "CRM",
    12: "CGM",
}

LABELS_ORDER = ["CRP", "CRM", "CRG", "CGP", "CGM", "CGG", "IPR", "IPG", "IM"]
LABELS = {
    "Congruent": "Congruent (C)",
    "Incongruent_Pur": "Incongruent Pur (IP)",
    "Incongruent_Mixt": "Incongruent Mixt (IM)",
    "Incongruent": "Incongruent (I)",
    "Errors": "Errors",
}
COLORS = {
    "Congruent": "steelblue",
    "Incongruent_Pur": "darkorange",
    "Incongruent_Mixt": "forestgreen",
    "Incongruent": "slategray",
    "Errors": "firebrick",
}

# Rangs temporals i bandes.
TRIAL_START = -0.74
TRIAL_END = 1.5
FULL_TRIAL = (TRIAL_START, TRIAL_END)
BASELINE = (-0.74, -0.24)
RAMP_UP = (-0.24, 0.0)
STIM_ONSET = (0.0, 1.5)
STIM_OFFSET = (1.5, 2.0)
ERP_WINDOW = (0.0, 0.8)

FINESTRA_1 = (0.0, 0.5)
FINESTRA_2 = (0.5, 1.0)
FINESTRA_3 = (1.0, 1.5)

FREQS_THETA = np.arange(5, 8)
FREQS_ALPHA = np.arange(8, 13)
FREQS_WIDE = np.arange(2, 21)

STFT_WINDOW_SEC = 0.5
STFT_STEP_SEC = 0.02

# Configuració específica de classificació.
TARGET_CHANNELS = {"theta": "FCz", "alpha": "Oz"}
TARGET_WINDOW = (1.0, 1.5)
ICA_MAX_COMPONENTS = 20

ROI_FRONTAL = ["Fz", "FCz", "Cz"]
ROI_PO = ["P7", "P8", "PO7", "PO3", "POz", "PO4", "PO8", "O1", "Oz", "O2"]

EYE_LEFT_CODES = [3, 6]
EYE_RIGHT_CODES = [4, 5]
COLOR_RED_CODES = [1, 3, 4, 10]
COLOR_GREEN_CODES = [2, 5, 6, 9]
ERROR_CODES = [9, 10, 11, 12]

ALPHA = 0.05
N_BINS = 3
BOOT_N = 2000
SMOOTH_WIN = 20

print("Inicialització compartida: constants carregades.")

In [ ]:
def load_subject(fif_path):
    ep = mne.read_epochs(fif_path, preload=True, verbose=False)
    for ch in ep.info["chs"]:
        ch["unit"] = mne.io.constants.FIFF.FIFF_UNIT_V
    ep.set_channel_types({ch: "eeg" for ch in ep.ch_names}, on_unit_change="ignore")

    cond = {}
    for cond_name, event_names in COND_EVENTS.items():
        existing = [n for n in event_names if n in ep.event_id]
        if existing:
            cond[cond_name] = ep[existing]
    return ep, cond


def count_trials_by_condition(fif_path, cond_events=COND_EVENTS):
    ep = mne.read_epochs(fif_path, preload=False, verbose=False)
    try:
        event_codes = ep.events[:, 2]
        event_id = ep.event_id
        counts = {}
        for cond_name, event_names in cond_events.items():
            codes = [event_id[n] for n in event_names if n in event_id]
            counts[cond_name] = int(np.isin(event_codes, codes).sum()) if codes else 0
        return counts
    finally:
        del ep


def row_binom_ci(successes, total):
    if int(total) <= 0:
        return np.nan, np.nan
    bt = binomtest(int(successes), n=int(total), p=0.5, alternative="two-sided")
    ci = bt.proportion_ci(confidence_level=0.95, method="exact")
    return ci.low, ci.high


def build_behavior_dominance_summary(data_dir=DATA_DIR):
    subject_rows = []
    comptes_classes = {cond: 0 for cond in COND_EVENTS}

    for fif_path in sorted(data_dir.glob("*.fif")):
        ep = mne.read_epochs(fif_path, preload=False, verbose=False)
        events = ep.events[:, 2]

        for cond_name, event_names in COND_EVENTS.items():
            codes = [ep.event_id[n] for n in event_names if n in ep.event_id]
            comptes_classes[cond_name] += int(np.isin(events, codes).sum()) if codes else 0

        counts = {k: 0 for k in LABELS_ORDER}
        for ev in events:
            if ev in EVENT_TO_LABEL:
                counts[EVENT_TO_LABEL[ev]] += 1

        left_eye = int(np.sum(np.isin(events, EYE_LEFT_CODES)))
        right_eye = int(np.sum(np.isin(events, EYE_RIGHT_CODES)))
        n_eye = left_eye + right_eye
        eye_index = (left_eye - right_eye) / n_eye if n_eye > 0 else np.nan
        p_eye = binomtest(left_eye, n=n_eye, p=0.5).pvalue if n_eye > 0 else np.nan
        ci_ull_low, ci_ull_high = row_binom_ci(left_eye, n_eye)

        red_count = int(np.sum(np.isin(events, COLOR_RED_CODES)))
        green_count = int(np.sum(np.isin(events, COLOR_GREEN_CODES)))
        n_color = red_count + green_count
        color_index = (red_count - green_count) / n_color if n_color > 0 else np.nan
        p_color = binomtest(red_count, n=n_color, p=0.5).pvalue if n_color > 0 else np.nan
        ci_color_low, ci_color_high = row_binom_ci(red_count, n_color)

        left_red = int(np.sum(events == 3))
        left_green = int(np.sum(events == 6))
        right_red = int(np.sum(events == 4))
        right_green = int(np.sum(events == 5))

        row = {
            "subjecte": fif_path.stem,
            **counts,
            "left_eye": left_eye,
            "right_eye": right_eye,
            "n_ull": n_eye,
            "index_dominancia_ull": eye_index,
            "p_ull": p_eye,
            "ci95_ull_low": ci_ull_low,
            "ci95_ull_high": ci_ull_high,
            "red_count": red_count,
            "green_count": green_count,
            "n_color": n_color,
            "index_dominancia_color": color_index,
            "p_color": p_color,
            "ci95_color_low": ci_color_low,
            "ci95_color_high": ci_color_high,
            "left_red": left_red,
            "left_green": left_green,
            "right_red": right_red,
            "right_green": right_green,
        }
        subject_rows.append(row)

    df = pd.DataFrame(subject_rows).sort_values("subjecte").reset_index(drop=True)
    label_counts = {k: int(df[k].sum()) for k in LABELS_ORDER}
    globals_summary = {
        "left_global": int(df["left_eye"].sum()),
        "right_global": int(df["right_eye"].sum()),
        "red_global": int(df["red_count"].sum()),
        "green_global": int(df["green_count"].sum()),
    }
    return df, comptes_classes, label_counts, globals_summary


def stft_power_db(ep, picks, freqs=FREQS_WIDE, baseline=BASELINE):
    x = ep.copy().pick(picks).get_data().astype(np.float32, copy=False)
    sfreq = ep.info["sfreq"]

    nperseg = int(round(STFT_WINDOW_SEC * sfreq))
    hop = int(round(STFT_STEP_SEC * sfreq))
    noverlap = max(0, nperseg - hop)
    nfft = max(nperseg, int(round(sfreq)))

    f, t, z = stft(
        x,
        fs=sfreq,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
        boundary="zeros",
        padded=True,
        axis=-1,
    )
    power = np.abs(z) ** 2

    freq_targets = np.asarray(freqs)
    freq_idx = np.array([np.argmin(np.abs(f - fr)) for fr in freq_targets])
    f_sel = f[freq_idx]
    power = power[:, :, freq_idx, :]

    times = ep.tmin + t
    bmask = (times >= baseline[0]) & (times <= baseline[1])
    b = power[..., bmask].mean(axis=-1, keepdims=True)
    power_db = 10 * np.log10((power + 1e-20) / (b + 1e-20))
    return f_sel, times, power_db


def stft_band_timecourse(ep, picks, band, baseline=BASELINE):
    f, times, power_db = stft_power_db(ep, picks=picks, freqs=FREQS_WIDE, baseline=baseline)
    fmask = (f >= band[0]) & (f <= band[1])
    band_db = power_db[:, :, fmask, :]
    mean_ts = band_db.mean(axis=(0, 1, 2))
    sem_ts = band_db.std(axis=(0, 1, 2)) / np.sqrt(band_db.shape[0])
    return times, mean_ts, sem_ts


def stft_condition_mean_db_chunked(ep, freqs=FREQS_WIDE, baseline=BASELINE, ch_block=4):
    ch_names = ep.ch_names
    n_ch = len(ch_names)
    acc = None
    f_ref, t_ref = None, None

    for start in range(0, n_ch, ch_block):
        picks = ch_names[start : start + ch_block]
        f, t, p_db = stft_power_db(ep, picks=picks, freqs=freqs, baseline=baseline)
        block_mean = p_db.mean(axis=(0, 1))

        if acc is None:
            acc = np.zeros_like(block_mean)
            f_ref, t_ref = f, t

        acc += block_mean * len(picks)
        del p_db, block_mean
        gc.collect()

    return f_ref, t_ref, acc / n_ch


def undersample_binary(x_in, y_in, random_state=42):
    classes, counts = np.unique(y_in, return_counts=True)
    if len(classes) < 2:
        return x_in, y_in, np.arange(len(y_in))

    n_min = counts.min()
    rng = np.random.default_rng(random_state)
    selected_idx = []
    for cls in classes:
        cls_idx = np.where(y_in == cls)[0]
        sampled = rng.choice(cls_idx, size=n_min, replace=False)
        selected_idx.extend(sampled.tolist())

    selected_idx = np.array(selected_idx)
    rng.shuffle(selected_idx)
    return x_in[selected_idx], y_in[selected_idx], selected_idx


print("Inicialització compartida: funcions reutilitzables carregades.")